# Attribution Analysis

Decompose the end-to-end speedup of `ours` over the `vllm` baseline into two
contributions:

- **GEMM optimization** — the GEMM cache. Measured by `onlyGEMM` runs, which
  apply the GEMM cache *but not* the dynamic chunk switch.
- **Attention optimization** — the dynamic chunk switch.

**Method (per beam setting `n`):** with average per-problem wall times
`A = vllm`, `B = ours`, `C = onlyGEMM`, the total time saved by `ours` is
`A - B`. The GEMM cache alone (no chunk switch) saves `A - C`, leaving `C - B`
for the chunk switch. This is an additive decomposition of the saved time:

- GEMM contribution `= (A - C) / (A - B)` (%)
- Attention contribution `= (C - B) / (A - B) = 1 - (A-C)/(A-B)` (%)

This requires the ordering `B <= C <= A` (`ours` fastest, `onlyGEMM` in
between) so both parts are non-negative.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ---- Configuration space ---------------------------------------------------
BEAMS    = [2, 4, 8, 16, 32, 64, 128]
DATASETS = ['MATH-500', 'AIME']

# ---- Paths -----------------------------------------------------------------
PROJECT_ROOT = Path('..').resolve()
MODEL_DIR    = 'Qwen/QWen2.5-1.5B-Instruct'

# Display name -> dataset folder under data/{MODEL_DIR}/.
DATASET_DIR = {
    'MATH-500': 'MATH-500',
    'AIME':     'aimo-validation-aime',
}

def csv_path(dataset: str, method: str, n: int) -> Path:
    """Locate a timing CSV.

    `vllm` and `ours` live under `<dataset>/best_of_n/`; the `onlyGEMM`
    ablation lives under `<dataset>/attribution_analysis/`.
    """
    base = PROJECT_ROOT / 'data' / MODEL_DIR / DATASET_DIR[dataset]
    filename = f'best_of_n_request_timings_{method}_n{n}.csv'
    if method == 'onlyGEMM':
        return base / 'attribution_analysis' / filename
    return base / 'best_of_n' / filename

## 1. Per-file aggregation

For one CSV: group rows by `problem`, take `max(gen_time_ms)` per problem (the
slowest beam = wall time for that problem), then average across all problems.

In [ ]:
def avg_wall_time_ms(path: Path) -> float:
    """Mean per-problem max gen_time_ms. NaN if the file is missing."""
    if not path.exists():
        return float('nan')
    df = pd.read_csv(path)
    problem_col = 'problem' if 'problem' in df.columns else 'problem_idx'
    per_problem_max = df.groupby(problem_col)['gen_time_ms'].max()
    return float(per_problem_max.mean())

## 2. Attribution table

For each beam setting, compute the wall times `A` (vllm), `B` (ours),
`C` (onlyGEMM), and split the total saved time `A - B` into a GEMM part
`A - C` and an attention part `C - B`.

In [ ]:
def attribution_table(dataset: str) -> pd.DataFrame:
    """Build a per-beam attribution table for one dataset.

    A = vllm, B = ours, C = onlyGEMM (average per-problem wall times, ms).
    Total saved = A - B; GEMM saves A - C, attention (chunk switch) saves C - B.
    """
    rows = []
    for n in BEAMS:
        A = avg_wall_time_ms(csv_path(dataset, 'vllm', n))
        B = avg_wall_time_ms(csv_path(dataset, 'ours', n))
        C = avg_wall_time_ms(csv_path(dataset, 'onlyGEMM', n))
        saved = A - B
        gemm_pct = (A - C) / saved * 100.0
        attn_pct = (C - B) / saved * 100.0
        rows.append({
            'beams':            n,
            'vllm_ms (A)':      A,
            'onlyGEMM_ms (C)':  C,
            'ours_ms (B)':      B,
            'saved (A-B)':      saved,
            'GEMM saved (A-C)': A - C,
            'Attn saved (C-B)': C - B,
            'GEMM %':           gemm_pct,
            'Attention %':      attn_pct,
        })
    return pd.DataFrame(rows).set_index('beams')

## 3. Run the analysis

In [ ]:
tables = {}
for dataset in DATASETS:
    tbl = attribution_table(dataset)
    tables[dataset] = tbl
    print(f'=== {dataset} | Qwen-1.5B | best_of_n — attribution ===')
    print(tbl.round(2).to_string())
    # Sanity check: ordering B <= C <= A (ours fastest, onlyGEMM in between).
    bad = tbl[(tbl['ours_ms (B)'] > tbl['onlyGEMM_ms (C)']) |
              (tbl['onlyGEMM_ms (C)'] > tbl['vllm_ms (A)'])]
    if len(bad):
        print(f'  WARNING: ordering B <= C <= A violated at beams {list(bad.index)}')
    print()

## 4. Stacked bar charts

Each bar sums to 100%. Lower segment = GEMM optimization contribution (`B/A`);
upper segment = attention optimization contribution (`1 - B/A`).

In [ ]:
import matplotlib.pyplot as plt

COLOR_GEMM = '#fac14f'
COLOR_ATTN = '#7fc4f5'

def plot_attribution(dataset: str, tbl: pd.DataFrame, savename: str):
    gemm = tbl['GEMM %'].values
    attn = tbl['Attention %'].values
    x = np.arange(len(BEAMS))

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(x, gemm, 0.6, color=COLOR_GEMM, label='GEMM optimization',
           edgecolor='black', linewidth=0.6)
    ax.bar(x, attn, 0.6, bottom=gemm, color=COLOR_ATTN,
           label='Attention optimization', edgecolor='black', linewidth=0.6)

    ax.set_ylim(0, 102)
    ax.set_xticks(x)
    ax.set_xticklabels([str(n) for n in BEAMS])
    ax.tick_params(axis='both', labelsize=14)
    ax.set_xlabel(r'#Beams ($\mathit{b}$)', fontsize=14)
    ax.set_ylabel('Contribution (%)', fontsize=14)
    ax.grid(True, axis='y', linestyle='--', alpha=0.8)
    ax.set_axisbelow(True)
#     ax.legend(fontsize=12)
    plt.tight_layout()
    (PROJECT_ROOT / 'figures').mkdir(exist_ok=True)
    fig.savefig(PROJECT_ROOT / 'figures' / savename, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_attribution('MATH-500', tables['MATH-500'],
                 'attribution_bon_MATH500_QWen.png')

In [ ]:
plot_attribution('AIME', tables['AIME'],
                 'attribution_bon_AIME_QWen.png')